# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice

I will use Logistic Regression as the first modeling method for this lane.

The task is to identify pages that are likely to need a refresh, so this can be treated as a binary classification problem: pages that should be prioritized for refresh versus pages that should not.

Logistic Regression is a suitable first model because it is simple, interpretable, and provides a useful comparison against the Week-4 rule-based baseline. It also allows the direction of feature effects to be inspected rather than relying only on a final score.

I am choosing a simple model first because the goal is to test whether learned signals improve on the existing decision rule, not to reward model complexity by itself.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split Design

The Week-4 baseline was a rule-based ranking method applied to the available dataset rather than a trained model with a train/test split.

For this modeling step, I will use a stratified train/test split so that the target classes are represented in both sets. The test set will be kept separate from model training and used only for final evaluation.

The split is intended to provide an honest check of whether the learned model can identify pages that need refresh beyond the rule-based baseline. No information from the test set will be used to fit the model or preprocessing steps.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [13]:
import os

print(os.path.exists("ml-internship-starter/data/raw/content_refresh_anonymized.csv"))

True


In [14]:
import pandas as pd

# Load the FlyRank dataset
df = pd.read_csv(
    "ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

# Create binary target
df["target_decline"] = (df["trend_direction"] == "down").astype(int)

# Check target distribution
print(df["target_decline"].value_counts())
print(df["target_decline"].value_counts(normalize=True))

target_decline
1    16262
0    13738
Name: count, dtype: int64
target_decline
1    0.542067
0    0.457933
Name: proportion, dtype: float64


In [15]:
# Show all dataset columns
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'target_decline']


In [16]:
from sklearn.model_selection import train_test_split

feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[feature_cols]
y = df["target_decline"]

# Stratified split so the class proportions remain similar
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

Training rows: 24000
Test rows: 6000

Training class distribution:
target_decline
1    0.542083
0    0.457917
Name: proportion, dtype: float64

Test class distribution:
target_decline
1    0.542
0    0.458
Name: proportion, dtype: float64


In [17]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Preprocessing + Logistic Regression
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# Train the model
model.fit(X_train, y_train)

# Predict on the held-out test set
y_pred = model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Logistic Regression Results")
print("---------------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Logistic Regression Results
---------------------------
Accuracy : 0.8225
Precision: 0.8468
Recall   : 0.8210
F1 Score : 0.8337


In [18]:
# Recreate the Week-4 baseline score
baseline_score = (
    (df["content_age_days"] > 365).astype(int) * 50
    + (df["trend_direction"] == "down").astype(int) * 40
    + (df["ctr"] < 0.02).astype(int) * 10
)

# Convert baseline action into a binary prediction
# 1 = Refresh, 0 = Monitor
baseline_pred = (baseline_score >= 50).astype(int)

# Use the SAME test rows as the Logistic Regression model
baseline_test_pred = baseline_pred.loc[y_test.index]

# Evaluate the baseline on the same test set
baseline_accuracy = accuracy_score(y_test, baseline_test_pred)
baseline_precision = precision_score(y_test, baseline_test_pred, zero_division=0)
baseline_recall = recall_score(y_test, baseline_test_pred, zero_division=0)
baseline_f1 = f1_score(y_test, baseline_test_pred, zero_division=0)

print("Week-4 Baseline Results")
print("-----------------------")
print(f"Accuracy : {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall   : {baseline_recall:.4f}")
print(f"F1 Score : {baseline_f1:.4f}")

Week-4 Baseline Results
-----------------------
Accuracy : 0.6165
Precision: 0.7005
Recall   : 0.5108
F1 Score : 0.5908


In [19]:
# Compare Week-4 baseline with Logistic Regression

comparison = pd.DataFrame({
    "Model": [
        "Week-4 Baseline",
        "Logistic Regression"
    ],
    "Accuracy": [
        baseline_accuracy,
        accuracy
    ],
    "Precision": [
        baseline_precision,
        precision
    ],
    "Recall": [
        baseline_recall,
        recall
    ],
    "F1": [
        baseline_f1,
        f1
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1
0,Week-4 Baseline,0.6165,0.700548,0.510763,0.590788
1,Logistic Regression,0.8225,0.846813,0.821033,0.833724


### Model vs Baseline

On the same held-out test set, Logistic Regression performed better than the Week-4 rule-based baseline across all measured metrics.

The observed F1 score increased from 0.5908 for the baseline to 0.8337 for Logistic Regression. Recall also increased from 0.5108 to 0.8210, meaning the learned model identified more of the observed declining pages in the test set.

These results are measured on this dataset and split only. They should be treated as directional evidence that the learned model provides stronger decision-support than the simple baseline, rather than as proof that it will generalize to every future dataset.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [20]:
from sklearn.metrics import confusion_matrix

# Confusion matrix for Logistic Regression
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix")
print("----------------")
print(cm)

print("\nError Summary")
print("-------------")
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

Confusion Matrix
----------------
[[2265  483]
 [ 582 2670]]

Error Summary
-------------
True Negatives : 2265
False Positives: 483
False Negatives: 582
True Positives : 2670


In [21]:
# Inspect Logistic Regression coefficients

coefficients = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.named_steps["classifier"].coef_[0]
})

# Largest positive coefficients
positive_features = coefficients.sort_values(
    "coefficient", ascending=False
).head(10)

# Largest negative coefficients
negative_features = coefficients.sort_values(
    "coefficient", ascending=True
).head(10)

print("Top features associated with the declining class")
print(positive_features.to_string(index=False))

print("\nTop features associated with the non-declining class")
print(negative_features.to_string(index=False))

Top features associated with the declining class
               feature  coefficient
  impressions_prev_30d    29.138803
       impressions_90d     1.259843
         pageviews_90d     1.045141
       clicks_prev_30d     0.955580
          sessions_90d     0.805672
 days_with_impressions     0.516423
            word_count     0.172616
days_since_last_update     0.120822
            clicks_90d     0.116106
  engaged_sessions_90d     0.054277

Top features associated with the non-declining class
             feature  coefficient
impressions_last_30d   -35.266653
           users_90d    -1.178345
     clicks_last_30d    -0.886946
   sessions_last_30d    -0.674286
  days_with_sessions    -0.407499
    content_age_days    -0.310618
        avg_position    -0.113943
          char_count    -0.103851
   sessions_prev_30d    -0.089406
                 ctr    -0.069440


### Error Analysis

On the held-out test set, the Logistic Regression model produced 2,265 true negatives, 2,670 true positives, 483 false positives, and 582 false negatives.

The 582 false negatives are pages that were observed as declining but were not identified by the model. The 483 false positives are pages that the model classified as declining even though they were not in the declining class. These errors show that the model is useful for decision-support but is not a perfect classifier.

### Feature Interpretation

The largest positive model coefficients were associated with `impressions_prev_30d`, `impressions_90d`, `pageviews_90d`, `clicks_prev_30d`, and `sessions_90d`. The strongest negative coefficients included `impressions_last_30d`, `users_90d`, `clicks_last_30d`, and `sessions_last_30d`.

These coefficients describe directional associations used by the model rather than causal effects. The results suggest that recent and historical traffic-related signals contain useful information for distinguishing observed declining pages in this dataset.

The remaining errors mean that the model should be treated as decision-support for prioritizing pages, not as an automatic refresh decision. Seasonal behavior, temporary search changes, or other signals not represented in the features could contribute to incorrect predictions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.